# Phase pickers across five earthquake sequences

Runs three PhaseNet weight sets over the **aftershocks** of five
sequences and compares what they recover. The question is whether the
fine-tuned weights hold up away from the one sequence they were first
checked on.

| Sequence | Date | M | Setting |
|---|---|--:|---|
| **Ridgecrest** | 2019-07-06 | 7.1 | Eastern California, dense aftershocks seconds apart |
| **San Simeon** | 2003-12-22 | 6.5 | Central Coast, 2003 network and instrumentation |
| **Monte Cristo** | 2020-05-15 | 6.5 | Nevada, Basin and Range, different network |
| **Mendocino** | 2024-12-05 | 7.0 | Offshore, one-sided geometry, every station 55 km+ |
| **Monroe WA** | 2019-07-12 | 4.6 | Cascadia, moderate magnitude |

Five stations each, chosen nearest-first among those that actually
return data — several catalogued stations serve nothing for these dates,
so the list was built by testing rather than from metadata.

### Weight sets

| | |
|---|---|
| `quakescope2026` | the v7 fine-tune, this project's production candidate |
| `jma_wc` | the SeisBench Japanese model v7 was distilled from — the baseline that matters |
| `original` | Zhu & Beroza (2019), the published reference |

### How it is scored

Four of the five sequences have published analyst arrivals, from SCEDC or
NCEDC, and those are the reference. **Recall is the metric; precision is
not computed.** Analyst catalogs are not exhaustive — especially in dense
aftershock sequences — so a model pick with no analyst counterpart may be
a false positive or a real arrival nobody had time to mark. Those are
reported as *extra detections* and never counted as errors.

Monroe WA has no published arrivals, so it is scored on **agreement
between the models** instead. That measures consistency, not correctness.

In [ ]:
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import obspy
import pandas as pd
import seisbench.models as sbm
from obspy import UTCDateTime
from obspy.clients.fdsn import Client

%matplotlib inline

## 1. Configuration

In [ ]:
# Five sequences, chosen to vary region, network, magnitude, and era.
# Stations were selected by testing that data actually returns, not from
# metadata alone - several catalogued stations serve nothing for these dates.
SEQUENCES = {
    "Ridgecrest": dict(
        time=UTCDateTime("2019-07-06T03:19:53"), lat=35.770, lon=-117.599, mag=7.1,
        picks_from="SCEDC", min_mag=3.2, window_min=30,
        stations=[("CI", "CLC", "HH"), ("CI", "TOW2", "HH"), ("CI", "SRT", "HH"),
                  ("CI", "WRC2", "HH"), ("CI", "JRC2", "HH")],
        note="dense aftershock sequence, events seconds apart",
    ),
    "San Simeon": dict(
        time=UTCDateTime("2003-12-22T19:15:56"), lat=35.700, lon=-121.100, mag=6.5,
        picks_from="NCEDC", min_mag=2.0, window_min=120,
        stations=[("BK", "PKD", "HH"), ("CI", "PHL", "HH"), ("CI", "SMM", "HH"),
                  ("BK", "SAO", "HH"), ("CI", "LCP", "HH")],
        note="2003 - sparser network; NCEDC published almost no S picks this era",
    ),
    "Monte Cristo": dict(
        time=UTCDateTime("2020-05-15T11:03:27"), lat=38.169, lon=-117.850, mag=6.5,
        picks_from="NCEDC", min_mag=2.0, window_min=120,
        stations=[("NN", "BRS2", "HH"), ("NN", "LHV", "HH"), ("NN", "ION4", "HH"),
                  ("NN", "DSP", "HH"), ("NN", "Q09A", "HH")],
        note="Nevada, Basin and Range - a different crust and a different network",
    ),
    "Mendocino 2024": dict(
        time=UTCDateTime("2024-12-05T18:44:21"), lat=40.374, lon=-125.022, mag=7.0,
        picks_from="NCEDC", min_mag=2.0, window_min=120,
        stations=[("BK", "PETL", "HH"), ("NC", "KCT", "HH"), ("BK", "WLKR", "HH"),
                  ("NC", "KMPB", "HH"), ("BK", "RBOW", "HH")],
        note="offshore - every station is one-sided and 55 km or more away",
    ),
    "Monroe WA": dict(
        time=UTCDateTime("2019-07-12T09:51:38"), lat=47.873, lon=-122.016, mag=4.6,
        picks_from=None, min_mag=1.0, window_min=30,
        stations=[("UW", "SP2", "HH"), ("UW", "BERY", "HH"), ("UW", "RATT", "HH"),
                  ("UW", "BST16", "HH"), ("UW", "BST20", "HH")],
        note="Cascadia, moderate magnitude - no analyst arrivals published, so "
             "this one is scored on model agreement only",
    ),
}

# Each network is served by its home archive. Asking the wrong one silently
# returns nothing, which is how station lists end up looking empty.
ROUTE = {"CI": "SCEDC", "BK": "NCEDC", "NC": "NCEDC", "NP": "NCEDC",
         "UW": "EARTHSCOPE", "NN": "EARTHSCOPE", "LB": "EARTHSCOPE",
         "IM": "EARTHSCOPE"}

WEIGHTS = ["quakescope2026", "jma_wc", "original"]

# Aftershock window starts after the mainshock coda. Its length is set per
# sequence, because aftershock productivity and how much of it the analysts
# worked through vary enormously - 30 minutes at Ridgecrest yields a larger
# reference than two hours at San Simeon.
WINDOW_START = 600      # seconds after origin, all sequences

P_THRESHOLD = S_THRESHOLD = 0.3
MATCH_TOL = 0.5         # seconds, for counting an analyst pick as recovered

COLORS = ["#2a78d6", "#eb6834", "#1baf7a"]
C_P, C_S = "#2a78d6", "#eb6834"


## 2. Data access

Waveforms come from FDSN rather than the S3 buckets, because these five
sequences span four archives and FDSN gives one interface to all of them.
Each network still has to be asked of its *home* archive — requesting CI
from EarthScope, or BK from SCEDC, returns nothing without raising.

In [ ]:
_clients = {}


def client_for(net):
    """FDSN client for whichever archive holds this network."""
    provider = ROUTE.get(net, "EARTHSCOPE")
    if provider not in _clients:
        _clients[provider] = Client(provider, timeout=300)
    return _clients[provider]


def fetch_sequence(seq):
    """Waveforms for every station of one sequence, over the aftershock window."""
    t0 = seq["time"] + WINDOW_START
    t1 = t0 + seq["window_min"] * 60
    streams = {}
    for net, sta, cha in seq["stations"]:
        try:
            st = client_for(net).get_waveforms(net, sta, "*", cha + "?", t0, t1)
        except Exception as exc:
            print(f"    {net}.{sta}: {type(exc).__name__}")
            continue
        st.merge(fill_value=0)
        expected = seq["window_min"] * 60 * st[0].stats.sampling_rate
        if len(st) < 3 or st[0].stats.npts < 0.5 * expected:
            print(f"    {net}.{sta}: incomplete")
            continue
        streams[f"{net}.{sta}"] = st
    return streams, t0, t1


def analyst_picks(seq, t0, t1):
    """Analyst P and S picks in the window, keyed by (station, phase)."""
    if seq["picks_from"] is None:
        return None
    keep = {f"{n}.{s}" for n, s, _ in seq["stations"]}
    try:
        cat = Client(seq["picks_from"], timeout=300).get_events(
            starttime=t0, endtime=t1, minmagnitude=seq["min_mag"],
            latitude=seq["lat"], longitude=seq["lon"], maxradius=1.0,
            includearrivals=True,
        )
    except Exception as exc:
        print(f"    catalog unavailable ({type(exc).__name__})")
        return None

    out = defaultdict(list)
    for ev in cat:
        origin = ev.preferred_origin() or (ev.origins[0] if ev.origins else None)
        if origin is None:
            continue
        by_id = {p.resource_id.id: p for p in ev.picks}
        for arr in origin.arrivals:
            pick = by_id.get(arr.pick_id.id)
            if pick is None or not arr.phase:
                continue
            phase = arr.phase[0].upper()
            if phase not in ("P", "S"):
                continue
            wid = pick.waveform_id
            key = f"{wid.network_code}.{wid.station_code}"
            if key in keep and t0 <= pick.time <= t1:
                out[(key, phase)].append(pick.time)
    return {k: sorted(v) for k, v in out.items()}


def match(reference, candidate, tol=MATCH_TOL):
    """Greedy nearest match; each candidate pick is consumed at most once."""
    used, residuals = set(), []
    for a in reference:
        best_i = best_d = None
        for i, m in enumerate(candidate):
            if i in used:
                continue
            d = m - a
            if abs(d) <= tol and (best_d is None or abs(d) < abs(best_d)):
                best_i, best_d = i, d
        if best_i is not None:
            used.add(best_i)
            residuals.append(best_d)
    return residuals, len(candidate) - len(used)


## 3. Load the weight sets

In [ ]:
available = sbm.PhaseNet.list_pretrained()
models = {}
for name in WEIGHTS:
    if name not in available:
        print(f"{name:<16} not installed - skipping")
        continue
    try:
        models[name] = sbm.PhaseNet.from_pretrained(name)
        print(f"{name:<16} loaded")
    except Exception as exc:
        print(f"{name:<16} could not load ({type(exc).__name__}) - skipping")

if not models:
    raise RuntimeError("no weight sets available")
names = list(models)

## 4. Run every model over every sequence

This is the slow cell — five sequences, five stations, three models, with
the waveforms fetched over the network. Expect several minutes.

In [ ]:
results = {}
for label, seq in SEQUENCES.items():
    print(f"{label} (M{seq['mag']}, {seq['note']})")
    streams, t0, t1 = fetch_sequence(seq)
    print(f"    {len(streams)}/{len(seq['stations'])} stations")
    if not streams:
        continue
    reference = analyst_picks(seq, t0, t1)
    if reference is not None:
        n_p = sum(len(v) for (_, ph), v in reference.items() if ph == 'P')
        n_s = sum(len(v) for (_, ph), v in reference.items() if ph == 'S')
        print(f"    analyst picks: {n_p} P, {n_s} S")
    else:
        print("    no analyst arrivals - model agreement only")

    picks = {}
    for name, model in models.items():
        per_station = defaultdict(list)
        for sta, st in streams.items():
            try:
                out = model.classify(st, P_threshold=P_THRESHOLD,
                                     S_threshold=S_THRESHOLD)
            except Exception as exc:
                print(f"    {name} {sta}: {type(exc).__name__}")
                continue
            for p in out.picks:
                per_station[(sta, p.phase)].append(p.peak_time)
        picks[name] = {k: sorted(v) for k, v in per_station.items()}
        tot = sum(len(v) for v in picks[name].values())
        print(f"    {name:<16} {tot} picks")

    results[label] = dict(streams=streams, picks=picks, reference=reference,
                          t0=t0, t1=t1)
    print()

## 5. Recall against analyst picks

S is the phase worth watching. It is the harder pick, it constrains depth
and location, and it is where the fine-tuning trade-offs surface.

In [ ]:
rows = []
for label, res in results.items():
    if res['reference'] is None:
        continue
    for name in names:
        if name not in res['picks']:
            continue
        for phase in ('P', 'S'):
            n_ref = n_hit = n_extra = 0
            residuals = []
            for sta in res['streams']:
                ref = res['reference'].get((sta, phase), [])
                got = res['picks'][name].get((sta, phase), [])
                r, extra = match(ref, got)
                n_ref += len(ref); n_hit += len(r); n_extra += extra
                residuals += r
            if n_ref == 0:
                continue
            rows.append(dict(
                sequence=label, weights=name, phase=phase,
                analyst=n_ref, matched=n_hit,
                recall=round(n_hit / n_ref, 3),
                MAE=round(float(np.mean(np.abs(residuals))), 3) if residuals else np.nan,
                extra=n_extra,
            ))

bench = pd.DataFrame(rows)
if len(bench):
    print(bench.to_string(index=False))
else:
    print('no analyst-referenced results')

In [ ]:
if len(bench):
    for phase in ('S', 'P'):
        sub = bench[bench.phase == phase]
        if not len(sub):
            continue
        piv = sub.pivot(index='sequence', columns='weights', values='recall')
        piv = piv.reindex(columns=[n for n in names if n in piv.columns])
        print(f'\n{phase} recall by sequence')
        print(piv.to_string())

### S recall, side by side

In [ ]:
if len(bench):
    sub = bench[bench.phase == 'S']
    seqs = [s for s in SEQUENCES if s in set(sub.sequence)]
    fig, ax = plt.subplots(figsize=(11, 4.4))
    width = 0.8 / max(len(names), 1)
    for i, name in enumerate(names):
        vals, xs = [], []
        for j, s in enumerate(seqs):
            row = sub[(sub.sequence == s) & (sub.weights == name)]
            if len(row):
                vals.append(float(row['recall'].iloc[0]))
                xs.append(j + (i - (len(names) - 1) / 2) * width)
        bars = ax.bar(xs, vals, width=width * 0.92, color=COLORS[i], label=name)
        for x, v in zip(xs, vals):
            ax.text(x, v + 0.015, f'{v:.2f}', ha='center', fontsize=8)
    ax.set_xticks(range(len(seqs)))
    ax.set_xticklabels(seqs, fontsize=9)
    ax.set_ylabel('S recall against analyst picks')
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.25, lw=0.5, axis='y')
    ax.set_axisbelow(True)
    ax.legend(frameon=False, fontsize=9, ncol=len(names))
    ax.set_title('S recall by sequence', fontsize=11, loc='left')
    fig.tight_layout()
    plt.show()

## 6. Monroe WA — agreement without a reference

With no published arrivals, the only thing measurable is whether the
models agree with each other. High agreement is not evidence of
correctness; disagreement is evidence that at least one is wrong.

In [ ]:
no_ref = [l for l, r in results.items() if r['reference'] is None]
for label in no_ref:
    res = results[label]
    print(f'{label}')
    for name in names:
        if name not in res['picks']:
            continue
        p = sum(len(v) for (_, ph), v in res['picks'][name].items() if ph == 'P')
        s = sum(len(v) for (_, ph), v in res['picks'][name].items() if ph == 'S')
        print(f'    {name:<16} {p:4d} P, {s:4d} S')
    print()
    pairs = [(a, b) for i, a in enumerate(names) for b in names[i + 1:]]
    rows = []
    for a, b in pairs:
        if a not in res['picks'] or b not in res['picks']:
            continue
        for phase in ('P', 'S'):
            shared = tot_a = 0
            for sta in res['streams']:
                pa = res['picks'][a].get((sta, phase), [])
                pb = res['picks'][b].get((sta, phase), [])
                r, _ = match(pa, pb)
                shared += len(r); tot_a += len(pa)
            rows.append(dict(pair=f'{a} vs {b}', phase=phase,
                             shared=shared, of_first=tot_a,
                             frac=round(shared / tot_a, 3) if tot_a else np.nan))
    print(pd.DataFrame(rows).to_string(index=False))
    print('\nfrac = fraction of the first model\'s picks that the second also found,'
          f' within {MATCH_TOL} s')

## 7. What the picks look like

One record section per sequence, from the model with the most picks, so
the numbers above can be checked against the waveforms they came from.
Aftershocks appear as separate moveout trains across the window.

In [ ]:
def record_section(label, res, weight, span=300):
    seq = SEQUENCES[label]
    fig, ax = plt.subplots(figsize=(11, 5))
    order = sorted(res['streams'],
                   key=lambda s: [f'{n}.{t}' for n, t, _ in seq['stations']].index(s)
                   if s in [f'{n}.{t}' for n, t, _ in seq['stations']] else 99)
    for row, sta in enumerate(order):
        tr = res['streams'][sta].select(component='Z')
        if not tr:
            continue
        tr = tr[0].copy()
        tr.trim(res['t0'], res['t0'] + span)
        x = tr.data.astype(float)
        peak = np.abs(x).max()
        if peak > 0:
            x = x / peak * 0.42
        ax.plot(tr.times(), x + row, color='#3d3d3d', lw=0.45)
        ax.annotate(sta, (-span * 0.015, row + 0.16), fontsize=8,
                    color='#52514e', ha='right')
        for (s2, phase), times in res['picks'][weight].items():
            if s2 != sta:
                continue
            for t in times:
                dt = t - res['t0']
                if 0 <= dt <= span:
                    ax.plot([dt, dt], [row - 0.34, row + 0.34],
                            color=C_P if phase == 'P' else C_S,
                            lw=1.1, ls='-' if phase == 'P' else '--', alpha=0.85)
    handles = [plt.Line2D([], [], color=C_P, lw=1.4, label='P'),
               plt.Line2D([], [], color=C_S, lw=1.4, ls='--', label='S')]
    ax.legend(handles=handles, frameon=False, fontsize=9, ncol=2, loc='upper right')
    ax.set_xlim(-span * 0.06, span)
    ax.set_ylim(-0.8, len(order) - 0.2)
    ax.set_yticks([])
    ax.set_xlabel(f'seconds into the aftershock window '
                  f'(starts {WINDOW_START // 60} min after the mainshock)')
    ax.set_title(f'{label} - M{seq["mag"]} - picks from {weight}',
                 fontsize=11, loc='left')
    ax.grid(alpha=0.2, lw=0.5, axis='x')
    fig.tight_layout()
    return fig


for label, res in results.items():
    best = max(res['picks'], key=lambda n: sum(len(v) for v in res['picks'][n].values()))
    record_section(label, res, best)
    plt.show()

## Reading the result

**Consistency across sequences matters more than any single number.** A
weight set that leads on Ridgecrest and collapses in Nevada is a worse
production choice than one that is second everywhere, because a campaign
runs over whatever the archives contain.

**The sequences are not equally difficult, by design.** Ridgecrest has
stations within 5 km; Mendocino is offshore with nothing closer than
55 km and one-sided geometry; San Simeon is a 2003 network. Absolute
recall is expected to differ between them, so read down each column, not
across.

**S counts vary enormously between catalogs, and one sequence has none.**
SCEDC publishes far more S picks than NCEDC does for comparable
sequences, and San Simeon has essentially no analyst S at these stations
— two picks across twelve hours, which is 2003 practice rather than
anything about the data. San Simeon therefore scores P only, and any S
recall computed against a handful of picks is far noisier than one
computed against several hundred. Always read the `analyst` column
alongside the recall; it is the sample size.

**Extra detections are not errors.** In a sequence producing hundreds of
events an hour, most unmatched picks are real earthquakes the analysts
never worked through. Separating those from false positives needs
association across stations, which is a different exercise.

For how these weights were selected and what that selection cost, see
[`docs/phasenet_v7_model_description.md`](../docs/phasenet_v7_model_description.md).